# Feux de forêt au Maroc : étapes 1 à 3
**Adam Talibi et Anouar Baoutoul — EMSI — 2026/2027**

Collecte et traçabilité, nettoyage, puis contrôle de la fusion météo.
Ce notebook utilise les sources figées du dossier `data`, sans nouvel appel API.
L’export FIRMS brut n’a pas été fourni : le capteur, les filtres de qualité et la construction des absences restent à documenter.
Exécuter depuis le dossier du projet après installation de `requirements.txt`.


## Étape 1 : charger les sources conservées


In [1]:
from pathlib import Path
import json
import hashlib
import platform
import pandas as pd
import numpy as np
import shapely
import pyproj
from shapely.geometry import shape, Point, box
from shapely.ops import unary_union

ROOT = Path.cwd()
DATA = ROOT / 'data'
OUT = ROOT / 'resultats'
OUT.mkdir(exist_ok=True)
source = DATA / 'dataset_feux_meteo_complete.csv'
raw = pd.read_csv(source, encoding='utf-8-sig')
original = raw.copy(deep=True)
required = ['lat_c','lon_c','acq_date','feu','mois','jour_annee','date',
            'temperature_2m_max','precipitation_sum','wind_speed_10m_max',
            'humidite_moy','lat_grille','lon_grille']
assert list(raw.columns) == required, 'Schéma inattendu : vérifier la source.'
raw.insert(0, 'id_observation', np.arange(1, len(raw)+1))
provenance = pd.read_csv(DATA/'provenance_meteo_retrouvee.csv')
recovered_ids = set(provenance['ligne_csv'].astype(int)-1)
raw['origine_meteo'] = np.where(raw.id_observation.isin(recovered_ids),
                               'recuperee_open_meteo','presente_dans_fichier_initial')
print(f'{len(raw):,} observations chargées ; {len(required)} colonnes sources.')
print('Source incendies déclarée : NASA FIRMS. Export satellitaire brut non fourni.')
print('Météo : archive Open-Meteo ; aucune nouvelle requête réseau dans ce notebook.')



22,788 observations chargées ; 13 colonnes sources.
Source incendies déclarée : NASA FIRMS. Export satellitaire brut non fourni.
Météo : archive Open-Meteo ; aucune nouvelle requête réseau dans ce notebook.


## Étape 2 : types, doublons, dates et cohérence physique


In [6]:
numeric = [c for c in required if c not in ('date','acq_date')]
for c in numeric:
    raw[c] = pd.to_numeric(raw[c], errors='raise')
assert np.isfinite(raw[numeric].to_numpy()).all(), 'Valeur numérique absente ou infinie.'
for c in ('date','acq_date'):
    raw[c] = pd.to_datetime(raw[c], format='%Y-%m-%d', errors='raise')
assert not raw[required].isna().any().any()
assert raw.feu.isin([0,1]).all()
for c in ('feu','mois','jour_annee'):
    assert (raw[c] == raw[c].round()).all()
    raw[c] = raw[c].astype('int64')
assert raw.lat_c.between(-90,90).all() and raw.lon_c.between(-180,180).all()
assert raw.lat_grille.between(-90,90).all() and raw.lon_grille.between(-180,180).all()
assert raw.date.eq(raw.acq_date).all(), 'Dates incendie/météo non concordantes.'
assert raw.mois.eq(raw.acq_date.dt.month).all()
assert raw.jour_annee.eq(raw.acq_date.dt.day_of_year).all()
assert raw.humidite_moy.between(0,100).all()
assert raw.precipitation_sum.ge(0).all() and raw.wind_speed_10m_max.ge(0).all()
assert raw.temperature_2m_max.gt(-273.15).all()
# Les extrêmes plausibles sont conservés ; pas de suppression automatique par IQR.
duplicate_count = int(raw.duplicated(required).sum())
clean = raw.drop_duplicates(required, keep='first').copy()
key = ['lat_c','lon_c','acq_date']
assert not clean.duplicated(key).any(), 'Clés contradictoires : examen manuel nécessaire.'
print(f'Doublons exacts retirés : {duplicate_count}. Valeurs manquantes : 0.')
print(clean[['temperature_2m_max','precipitation_sum','wind_speed_10m_max','humidite_moy']].agg(['min','max']).to_string())



Doublons exacts retirés : 0. Valeurs manquantes : 0.
     temperature_2m_max  precipitation_sum  wind_speed_10m_max  humidite_moy
min                -0.7                0.0                 3.3      7.666667
max                48.0               69.5                56.3     99.916667


## Étape 2 : contrôle spatial tenant compte de la précision des coordonnées


In [7]:
geojson = json.loads((DATA/'maroc.geojson').read_text(encoding='utf-8'))
country = unary_union([shape(f['geometry']) for f in geojson['features']])
assert country.is_valid, 'Contour invalide : ne pas réparer sans examiner la géométrie.'
coords = clean[['lat_c','lon_c']].drop_duplicates().copy()
# Les coordonnées sont toutes quantifiées au dixième de degré.
assert np.allclose(coords.to_numpy()*10, np.round(coords.to_numpy()*10))
def geo_status(lat, lon):
    if country.covers(Point(lon, lat)):
        return 'centre_dans_contour'
    # Hypothèse de précision : ±0,05°. Ce recouvrement ne prouve pas l'origine du feu.
    if country.intersects(box(lon-.05,lat-.05,lon+.05,lat+.05)):
        return 'limite_a_verifier'
    return 'hors_emprise_a_verifier'
coords['statut_geo'] = [geo_status(lat,lon) for lat,lon in coords.itertuples(index=False,name=None)]
clean = clean.merge(coords, on=['lat_c','lon_c'], how='left', validate='many_to_one')
assert not clean.statut_geo.isna().any()
# Les cas ambigus sont conservés et isolés pour examen ; aucune relocalisation fictive.
border = clean[clean.statut_geo.ne('centre_dans_contour')].copy()
border.to_csv(OUT/'points_limites_a_verifier.csv', index=False, encoding='utf-8-sig',date_format='%Y-%m-%d')
print(clean.statut_geo.value_counts().to_string())
print('Cas limites conservés : demander les coordonnées satellitaires non arrondies pour confirmer.')



statut_geo
centre_dans_contour    22219
limite_a_verifier        569
Cas limites conservés : demander les coordonnées satellitaires non arrondies pour confirmer.


## Étape 3 : reconstruire les deux tables et vérifier une fusion exacte


In [8]:
# La source livrée est déjà fusionnée. Cette séparation permet de contrôler les clés,
# mais ne constitue pas une validation indépendante des fichiers satellitaires bruts.
obs_cols = ['id_observation','lat_c','lon_c','acq_date','feu','mois','jour_annee','statut_geo']
meteo_cols = ['lat_c','lon_c','date','temperature_2m_max','precipitation_sum',
              'wind_speed_10m_max','humidite_moy','lat_grille','lon_grille','origine_meteo']
observations = clean[obs_cols].copy()
meteo = clean[meteo_cols].copy()
assert not meteo.duplicated(['lat_c','lon_c','date']).any()
merged = observations.merge(meteo, how='left', left_on=key,
                            right_on=['lat_c','lon_c','date'], validate='one_to_one',indicator=True)
assert merged['_merge'].eq('both').all()
assert len(merged)==len(observations)
merged = merged.drop(columns='_merge').sort_values('id_observation').reset_index(drop=True)
geod = pyproj.Geod(ellps='WGS84')
_,_,dist = geod.inv(merged.lon_c.to_numpy(),merged.lat_c.to_numpy(),
                   merged.lon_grille.to_numpy(),merged.lat_grille.to_numpy())
merged['distance_grille_km'] = dist/1000
# La distance entre grille météo et point de requête est normale : conserver les deux.
merged = merged[['id_observation']+required+['origine_meteo','statut_geo','distance_grille_km']]
assert not merged.isna().any().any()
check = clean.sort_values('id_observation').reset_index(drop=True)
pd.testing.assert_frame_equal(check[required],merged[required],check_dtype=False)
observations.to_csv(OUT/'observations_nettoyees.csv',index=False,encoding='utf-8-sig',date_format='%Y-%m-%d')
meteo.to_csv(OUT/'meteo_nettoyee.csv',index=False,encoding='utf-8-sig',date_format='%Y-%m-%d')
merged.to_csv(OUT/'dataset_prepare.csv',index=False,encoding='utf-8-sig',date_format='%Y-%m-%d')
print(f'Fusion exacte : {len(merged):,}/{len(observations):,}, soit 100 %.')
print(f'Distance maximale point/grille météo : {merged.distance_grille_km.max():.2f} km.')



Fusion exacte : 22,788/22,788, soit 100 %.
Distance maximale point/grille météo : 12.47 km.


## Bilan reproductible et vérification des fichiers exportés


In [9]:
report = {
    'lignes_source':len(original),'lignes_preparees':len(merged),
    'doublons_exacts_supprimes':duplicate_count,'valeurs_manquantes':int(merged.isna().sum().sum()),
    'dates_min_max':[str(merged.acq_date.min().date()),str(merged.acq_date.max().date())],
    'classes':{str(k):int(v) for k,v in merged.feu.value_counts().items()},
    'controle_geographique':{str(k):int(v) for k,v in merged.statut_geo.value_counts().items()},
    'lignes_meteo_retrouvees':int(merged.origine_meteo.eq('recuperee_open_meteo').sum()),
    'couverture_fusion':1.0,'distance_grille_max_km':float(merged.distance_grille_km.max()),
    'sha256_source':hashlib.sha256(source.read_bytes()).hexdigest(),
    'versions':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,
                'shapely':shapely.__version__,'pyproj':pyproj.__version__},
    'limites':['Export FIRMS/VIIRS brut et critères de qualité non fournis.',
               'Construction des exemples feu=0 non documentée.',
               'Points arrondis proches des limites à confirmer avec les coordonnées initiales.',
               'Absence de surface brûlée : régression de superficie non réalisable avec cette table seule.',
               'Météo du jour : ne pas utiliser comme information déjà connue au début de ce jour.']
}
(OUT/'rapport_controles.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
saved = pd.read_csv(OUT/'dataset_prepare.csv',encoding='utf-8-sig')
assert len(saved)==len(merged) and not saved.isna().any().any()
assert list(saved.columns)==list(merged.columns)
assert saved.id_observation.is_unique
for c in numeric:
    assert np.allclose(saved[c],merged[c],rtol=0,atol=1e-10)
print(json.dumps(report,ensure_ascii=False,indent=2))


{
  "lignes_source": 22788,
  "lignes_preparees": 22788,
  "doublons_exacts_supprimes": 0,
  "valeurs_manquantes": 0,
  "dates_min_max": [
    "2018-01-01",
    "2025-12-31"
  ],
  "classes": {
    "0": 14761,
    "1": 8027
  },
  "controle_geographique": {
    "centre_dans_contour": 22219,
    "limite_a_verifier": 569
  },
  "lignes_meteo_retrouvees": 21106,
  "couverture_fusion": 1.0,
  "distance_grille_max_km": 12.473384026420256,
  "sha256_source": "d9da9116b268b7c891a92c33cf7837e3bf91be3a087346bf86753bbe5474db7c",
  "versions": {
    "python": "3.13.14",
    "pandas": "3.1.0",
    "numpy": "2.5.4",
    "shapely": "2.2.0",
    "pyproj": "3.8.0"
  },
  "limites": [
    "Export FIRMS/VIIRS brut et critères de qualité non fournis.",
    "Construction des exemples feu=0 non documentée.",
    "Points arrondis proches des limites à confirmer avec les coordonnées initiales.",
    "Absence de surface brûlée : régression de superficie non réalisable avec cette table seule.",
    "Météo du j

## Suite du projet
4. Définir précisément l’horizon de prévision (par exemple J+1), documenter les labels, puis construire les variables disponibles avant cette échéance.
5. Obtenir les séries météo quotidiennes continues avant de calculer les moyennes sur les sept jours précédents. Sept observations espacées ne représentent pas sept jours.
6. Préparer une séparation temporelle : entraînement 2018–2023, validation 2024, test 2025, avec contrôle supplémentaire de généralisation spatiale.
7. Comparer un modèle de référence, une régression logistique et un Random Forest. Mesurer notamment rappel, précision et PR-AUC. Les classes échantillonnées ne donnent pas directement une probabilité réelle de feu.
8. Sauvegarder le prétraitement et le modèle validés, puis créer l’API de prédiction.
9. Développer l’interface et la carte ; utiliser Vercel une fois le fonctionnement du modèle et de son API vérifié.
La régression de la superficie nécessite une cible de surface brûlée, absente du fichier actuel.
